## Implementing Row-Level and Column-Level Security

In [0]:
%sql
-- Configuration
USE CATALOG workspace;
USE SCHEMA gold


In [0]:
%sql
SELECT * FROM consumption_hourly LIMIT 5

In [0]:
%sql
-- Creating a regional group
CREATE GROUP Poland WITH USER `gabrielajaniszewska@translite.pl`;

In [0]:
%sql
SHOW GROUPS WITH USER `gabrielajaniszewska@translite.pl`;

### Granting permission to objects

In [0]:
%sql
GRANT SELECT ON TABLE consumption_hourly TO `account users`;

### Regional Row Filter

In [0]:
%sql
-- Creating regional filter
CREATE OR REPLACE FUNCTION regional_filter(bidding_zone STRING)
RETURNS BOOLEAN
RETURN 
  CASE
    WHEN is_member('Poland') AND is_member('admins') AND bidding_zone = 'PL' THEN TRUE -- Users from Poland can see all data from Poland
    WHEN is_member('Poland') AND is_member('admins') AND bidding_zone != 'PL' THEN FALSE -- Users from Poland can't see data from other countries
    ELSE FALSE -- Users from other countries can't see data from Poland
    END

In [0]:
%sql
-- Altering materialized view to apply the regional filter
ALTER MATERIALIZED VIEW consumption_hourly
SET ROW FILTER regional_filter ON (bidding_zone);

In [0]:
%sql
-- Checking the filter - should return only data from Poland
SELECT bidding_zone, AVG(cost_per_hour) FROM consumption_hourly GROUP BY bidding_zone LIMIT 10

In [0]:
%sql
ALTER MATERIALIZED VIEW consumption_hourly
DROP ROW FILTER;

### Column Level Security - masking site IDs

In [0]:
%sql
CREATE OR REPLACE FUNCTION site_id_mask(site_id STRING)
RETURNS STRING
RETURN
    CASE
    WHEN is_member('Poland') AND site_id LIKE 'DC-PL-%' THEN site_id -- Poland: PL sites visible
    WHEN is_member('Poland') THEN '**-**-**' -- the rest is masked
    ELSE  site_id -- other users see everything
END

In [0]:
%sql
ALTER MATERIALIZED VIEW consumption_hourly
ALTER COLUMN site_id SET MASK site_id_mask;

In [0]:
%sql
-- Checking the filter - should mask site_ids from other countries
SELECT site_id, bidding_zone, AVG(cost_per_hour) FROM consumption_hourly GROUP BY bidding_zone, site_id LIMIT 10

Applying this filter impacts the dashboard:

![image_1787416316234.png](./image_1787416316234.png "image_1787416316234.png")

In [0]:
%sql
-- Something that surprised me at first -- grouping by masked column creates only two groups - one for Poland and one for the rest (all remaining site IDs are collapsed into one group and undistinguishable, which is actually the goal of the masking)
SELECT site_id, AVG(cost_per_hour) FROM consumption_hourly GROUP BY site_id LIMIT 10

In [0]:
%sql
ALTER MATERIALIZED VIEW consumption_hourly
ALTER COLUMN site_id DROP MASK;